# 01 — Time Series Understanding and Exploration

## 1. Study Context and Univariate Forecasting Objective

This repository is being developed as a standalone scientific study of the `nottem` dataset to establish a reproducible reference workflow for **univariate time-series forecasting** before any forecasting capability is implemented in Atlas DataFlow.

The study treats forecasting as a temporally ordered prediction problem rather than as ordinary tabular continuous regression. The time axis is part of the scientific identity of each observation, so all later preparation, validation, model selection, final evaluation, and inference decisions must preserve temporal causality and prevent future observations from influencing earlier fitted operations or evaluations.

The objective of this notebook is to build the evidence needed to define a defensible forecasting contract from the source series itself. The study is intentionally constrained to a single endogenous target series: forecasting must be based on its observed history, without introducing exogenous predictors that are not part of the original source.

The existing repository was adapted from a continuous-regression study. Any inherited regression-specific code, contracts, metrics, models, artifacts, documentation, or assumptions are therefore structural references only and are not evidence for the scientific design of this forecasting study.

At this stage, the following remain intentionally open and must be authenticated or decided only in their dedicated sections:

- source identity and reproducible Python acquisition;
- source time-series representation and canonical temporal index;
- observed frequency, temporal coverage, continuity, and target unit;
- trend, seasonality, lag dependence, stationarity signals, anomalies, and structural changes;
- forecast horizon and final-holdout boundary;
- backtesting design and forecasting-origin semantics;
- baseline definitions and model families;
- primary and secondary forecasting metrics; and
- the exact machine-readable forecasting contract and downstream artifact semantics.

This notebook is exploratory and contractual only. It does not implement Atlas integration, fit forecasting models, perform model selection, open a final holdout, or define downstream production behavior.

## 2. Dataset Source and Python Acquisition

In [1]:
from IPython.display import display
import json

import pandas as pd

from scripts.download_data import acquire_rdataset


DATASET_NAME = "nottem"
R_PACKAGE = "datasets"
RAW_DATA_DIR = "data/raw/nottem"

acquisition = acquire_rdataset(
    dataset_name=DATASET_NAME,
    package=R_PACKAGE,
    destination=RAW_DATA_DIR,
)

data_path = acquisition.require_one_file("dataset.csv")
metadata_path = acquisition.require_one_file("metadata.json")
documentation_path = acquisition.require_one_file("documentation.txt")

source_data = pd.read_csv(data_path)
source_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

print(f"Source type: {acquisition.source_kind}")
print(f"Source reference: {acquisition.source_reference}")
print(f"Destination: {acquisition.display_destination}")
print(f"Materialized files: {list(acquisition.relative_files)}")
print(f"Source title: {source_metadata['title']}")
print(f"Raw shape: {source_data.shape}")
print(f"Raw columns: {source_data.columns.tolist()}")
print(f"Documentation file: {documentation_path.name}")

display(source_data.head())

Source type: rdataset
Source reference: R dataset datasets::nottem
Destination: data/raw/nottem
Materialized files: ['data/raw/nottem/dataset.csv', 'data/raw/nottem/documentation.txt', 'data/raw/nottem/metadata.json']
Source title: Average Monthly Temperatures at Nottingham, 1920-1939
Raw shape: (240, 2)
Raw columns: ['time', 'value']
Documentation file: documentation.txt


,time,value
0,1920.000000,40.6
1,1920.083333,40.8
2,1920.166667,44.4
3,1920.250000,46.7
4,1920.333333,54.1


## 3. Source Time-Series Representation and Temporal Observation Identity

The scientific source is the R object `datasets::nottem`, documented as a **time-series object** containing average air temperatures at Nottingham Castle in degrees Fahrenheit. Its title identifies the observations as monthly averages. The source is therefore scientifically univariate: one temporally ordered scalar temperature measurement is associated with each source time coordinate, and no exogenous predictor series is part of the original dataset.

The Python acquisition layer materializes that source as a two-column table named `time` and `value`. This tabular transport representation does **not** redefine the problem as tabular regression and does not make `time` an ordinary predictive feature. Instead:

- `time` is the source-series temporal coordinate carried through the Rdatasets representation;
- `value` is the single observed endogenous measurement;
- row order preserves the source-series order;
- one row represents one monthly average air-temperature observation at Nottingham Castle; and
- there are no source exogenous predictors to add to the forecasting input contract.

The acquired `time` values are numeric fractional-year coordinates rather than authenticated calendar timestamps. They must therefore be preserved as source coordinates at this stage, not silently converted to arbitrary dates. The source row position together with the raw `time` coordinate provides the provisional observation identity until the canonical monthly index is reconstructed and validated in the next section.

The R `ts` semantics also do not imply a timezone-bearing instant. A monthly average is a calendar-period observation, not an event timestamp. Exact calendar labeling, frequency verification, start/end coverage, continuity, and timezone applicability are intentionally deferred to the canonical time-index reconstruction step.

This distinction is contractual: the **target value** is the observed temperature measurement, while forecast horizon, forecast origin, and allowed historical context are separate forecasting concepts that remain undefined at this point.

In [2]:
EXPECTED_SOURCE_COLUMNS = ("time", "value")
EXPECTED_SOURCE_REFERENCE = "datasets::nottem"

observed_columns = tuple(source_data.columns)

if observed_columns != EXPECTED_SOURCE_COLUMNS:
    raise ValueError(
        "Unexpected source representation: "
        f"expected columns {EXPECTED_SOURCE_COLUMNS}, observed {observed_columns}."
    )

if source_metadata.get("source_reference") != EXPECTED_SOURCE_REFERENCE:
    raise ValueError(
        "Unexpected source identity: "
        f"expected {EXPECTED_SOURCE_REFERENCE!r}, "
        f"observed {source_metadata.get('source_reference')!r}."
    )

if not pd.api.types.is_numeric_dtype(source_data["time"]):
    raise TypeError("The raw R time coordinate must remain numeric at this stage.")

if not pd.api.types.is_numeric_dtype(source_data["value"]):
    raise TypeError("The source value column must be numeric.")

source_representation = pd.Series(
    {
        "scientific_source": "datasets::nottem",
        "source_object_semantics": "univariate R time-series object",
        "python_materialization": "two-column table: time, value",
        "raw_time_role": "source temporal coordinate; not a predictive feature",
        "raw_time_representation": "numeric fractional-year coordinate",
        "observed_value_role": "single endogenous temperature measurement",
        "temporal_observation_unit": (
            "one monthly average air-temperature observation at Nottingham Castle"
        ),
        "source_exogenous_predictors": 0,
        "canonical_calendar_index": "not reconstructed yet",
        "timezone_semantics": "not assigned; applicability still to be confirmed",
    },
    name="source representation",
)

display(source_representation.to_frame())
display(source_data.loc[:, list(EXPECTED_SOURCE_COLUMNS)].head())

,source representation
scientific_source,datasets::nottem
source_object_semantics,univariate R time-series object
python_materialization,"two-column table: time, value"
raw_time_role,source temporal coordinate; not a predictive f...
raw_time_representation,numeric fractional-year coordinate
observed_value_role,single endogenous temperature measurement
temporal_observation_unit,one monthly average air-temperature observatio...
source_exogenous_predictors,0
canonical_calendar_index,not reconstructed yet
timezone_semantics,not assigned; applicability still to be confirmed


,time,value
0,1920.000000,40.6
1,1920.083333,40.8
2,1920.166667,44.4
3,1920.250000,46.7
4,1920.333333,54.1


## 4. Canonical Time Index Reconstruction, Frequency, and Temporal Coverage

The authenticated source semantics identify `datasets::nottem` as monthly average temperatures covering 1920–1939. The acquired Python table carries the original R time-series coordinate as fractional years, so this section converts that source coordinate into an explicit calendar-period index without inventing day-level or timezone-bearing timestamps.

A **monthly `pandas.PeriodIndex`** is the canonical temporal representation for this study. A calendar month is the scientific observation period, whereas a `DatetimeIndex` would require an arbitrary day or instant that is not part of the source semantics. Timezone assignment is therefore not applicable to the canonical index.

The reconstruction follows the R `ts` coordinate semantics for a monthly series:

- the integer component of `time` identifies the calendar year;
- the fractional component is converted to a zero-based month position using 12 source periods per year;
- the reconstructed periods must align with the authenticated source coverage from `1920-01` through `1939-12`;
- the resulting canonical index must contain exactly 240 monthly periods; and
- the numerical fractional-year representation is accepted only when it is sufficiently close to the expected monthly grid.

The source frequency is therefore frozen as **monthly, 12 observations per year**. This frequency defines the calendar spacing of the observations; it does not by itself prove seasonal dependence or justify a particular forecasting horizon. Seasonal strength and lag dependence remain subjects of later exploratory sections.

The canonical series constructed here preserves the acquired values exactly and changes only their temporal labeling. No resampling, interpolation, aggregation, smoothing, differencing, seasonal adjustment, imputation, or forecasting feature engineering is performed.

Regularity is validated here only as required to authenticate the reconstruction and coverage. Dedicated diagnostics for missing periods, duplicate timestamps, ordering integrity, and related data-quality conditions remain in their later sections.

In [3]:
import numpy as np


EXPECTED_FREQUENCY = "M"
SOURCE_PERIODS_PER_YEAR = 12
EXPECTED_START_PERIOD = pd.Period("1920-01", freq=EXPECTED_FREQUENCY)
EXPECTED_END_PERIOD = pd.Period("1939-12", freq=EXPECTED_FREQUENCY)
EXPECTED_OBSERVATIONS = 240
TIME_COORDINATE_TOLERANCE = 1e-8

raw_time = source_data["time"].to_numpy(dtype=float, copy=True)

source_year = np.floor(raw_time).astype(int)
source_month_position = (raw_time - source_year) * SOURCE_PERIODS_PER_YEAR
source_month_offset = np.rint(source_month_position).astype(int)

max_coordinate_residual = float(
    np.max(np.abs(source_month_position - source_month_offset))
)

if max_coordinate_residual > TIME_COORDINATE_TOLERANCE:
    raise ValueError(
        "The raw time coordinate is not aligned with authenticated monthly "
        "R ts semantics within tolerance: "
        f"max residual={max_coordinate_residual:.3e}."
    )

if np.any(
    (source_month_offset < 0)
    | (source_month_offset >= SOURCE_PERIODS_PER_YEAR)
):
    raise ValueError("The raw time coordinate produced an invalid month offset.")

canonical_period_index = pd.PeriodIndex.from_fields(
    year=source_year,
    month=source_month_offset + 1,
    freq=EXPECTED_FREQUENCY,
).rename("period")

expected_period_index = pd.period_range(
    start=EXPECTED_START_PERIOD,
    end=EXPECTED_END_PERIOD,
    freq=EXPECTED_FREQUENCY,
    name="period",
)

if len(canonical_period_index) != EXPECTED_OBSERVATIONS:
    raise ValueError(
        "Unexpected observation count for the authenticated source coverage: "
        f"expected {EXPECTED_OBSERVATIONS}, "
        f"observed {len(canonical_period_index)}."
    )

if not canonical_period_index.equals(expected_period_index):
    raise ValueError(
        "The reconstructed canonical period index does not match the "
        "authenticated monthly coverage from 1920-01 through 1939-12."
    )

canonical_source_series = pd.Series(
    source_data["value"].to_numpy(copy=True),
    index=canonical_period_index,
    name="value",
)

temporal_coverage = pd.Series(
    {
        "canonical_index_type": "pandas.PeriodIndex",
        "frequency": "monthly",
        "pandas_frequency": canonical_period_index.freqstr,
        "source_periods_per_year": SOURCE_PERIODS_PER_YEAR,
        "start_period": str(canonical_period_index[0]),
        "end_period": str(canonical_period_index[-1]),
        "observation_count": len(canonical_period_index),
        "coverage_years": 20,
        "timezone": "not applicable to monthly PeriodIndex",
        "max_fractional_year_reconstruction_residual": (
            max_coordinate_residual
        ),
    },
    name="temporal coverage",
)

display(temporal_coverage.to_frame())
display(canonical_source_series.head().to_frame())
display(canonical_source_series.tail().to_frame())

,temporal coverage
canonical_index_type,pandas.PeriodIndex
frequency,monthly
pandas_frequency,M
source_periods_per_year,12
start_period,1920-01
end_period,1939-12
observation_count,240
coverage_years,20
timezone,not applicable to monthly PeriodIndex
max_fractional_year_reconstruction_residual,0.0


,value
period,
1920-01,40.6
1920-02,40.8
1920-03,44.4
1920-04,46.7
1920-05,54.1


,value
period,
1939-08,61.8
1939-09,58.2
1939-10,46.7
1939-11,46.6
1939-12,37.8


## 5. Target and Univariate Forecasting Contract

The forecasting target is canonically named **`temperature`** and represents the **monthly average air temperature at Nottingham Castle**, measured in **degrees Fahrenheit**. The acquired source column remains `value`; assigning the semantic target name does not alter, transform, aggregate, or rescale any observation.

This study freezes the task as:

- `problem_type`: `time_series_forecasting`;
- `forecasting_mode`: `univariate`;
- one endogenous target series: `temperature`;
- canonical temporal identity: monthly `pandas.PeriodIndex`;
- source frequency: 12 observations per year (`M`);
- source exogenous predictors: none; and
- forecast scale: the original temperature scale in degrees Fahrenheit.

For this study, **univariate** means that the quantity being forecast is a single endogenous series. The deterministic future calendar index implied by the monthly frequency is part of the temporal contract, not an exogenous predictor. No source-provided future covariate series exists.

The forecasting objective is to estimate future values of `temperature` after a forecast origin using only information permitted by the eventual historical-data contract. Forecasted values must remain associated with explicit future monthly periods and preserve the target unit unless a later modeling transformation is explicitly introduced and correctly inverted for public output.

The following concepts are deliberately **not frozen here** because target identity and forecasting geometry are separate concerns:

- forecast horizon;
- forecast origin;
- minimum or maximum allowed history;
- expanding-window or rolling-window evaluation design;
- final-holdout boundary;
- recursive, direct, or other multi-step forecasting strategy;
- baseline definition;
- model family;
- target transformation or differencing policy;
- primary and secondary evaluation metrics; and
- prediction-interval or uncertainty semantics.

Those decisions require evidence from the remaining exploration and the later preparation/evaluation-design notebook. In particular, monthly frequency does not imply that the forecast horizon must be 12 months.

The reusable target-contract helper validates only the structural invariants justified at this point: a numeric endogenous `pandas.Series`, a canonical `PeriodIndex`, the declared frequency, the forecasting problem family, and its univariate mode. Missingness, non-finite values, duplicates, distribution, range, anomalies, seasonal behavior, and forecastability remain dedicated downstream checks.

In [4]:
from scripts.target_contract import define_univariate_forecasting_target_contract


TARGET_NAME = "temperature"
TARGET_SEMANTICS = "Monthly average air temperature at Nottingham Castle"
TARGET_UNIT = "degrees Fahrenheit"
PROBLEM_TYPE = "time_series_forecasting"
FORECASTING_MODE = "univariate"

forecasting_target_contract = define_univariate_forecasting_target_contract(
    canonical_source_series,
    target=TARGET_NAME,
    problem_type=PROBLEM_TYPE,
    forecasting_mode=FORECASTING_MODE,
    target_semantics=TARGET_SEMANTICS,
    target_unit=TARGET_UNIT,
    expected_frequency=EXPECTED_FREQUENCY,
    source_exogenous_predictors=0,
)

target_series = canonical_source_series.rename(TARGET_NAME)

if not target_series.index.equals(canonical_source_series.index):
    raise ValueError("Canonical target naming must not alter the temporal index.")

if not np.array_equal(
    target_series.to_numpy(),
    canonical_source_series.to_numpy(),
    equal_nan=True,
):
    raise ValueError("Canonical target naming must not alter source values.")

open_forecasting_decisions = pd.Series(
    {
        "forecast_horizon": "open",
        "forecast_origin": "open",
        "allowed_history": "open",
        "backtesting_design": "open",
        "final_holdout_boundary": "open",
        "multi_step_strategy": "open",
        "baseline": "open",
        "model_family": "open",
        "target_transformation": "open",
        "evaluation_metrics": "open",
        "uncertainty_semantics": "open",
    },
    name="status",
)

display(forecasting_target_contract.summary_frame())
display(open_forecasting_decisions.to_frame())
display(target_series.head().to_frame())

,Contract item,Value,Interpretation
0,Problem type,time_series_forecasting,Temporally ordered forecasting task
1,Forecasting mode,univariate,One endogenous target series
2,Canonical target,temperature,Quantity to be forecast at future periods
3,Source value column,value,Raw acquired value representation
4,Target semantics,Monthly average air temperature at Nottingham ...,Scientific meaning of each target value
5,Target unit,degrees Fahrenheit,Original target and forecast scale
6,Temporal index,PeriodIndex,Canonical observation-period identity
7,Frequency,M,Canonical spacing of forecast periods
8,Source exogenous predictors,0,Exogenous series present in the source data
9,Prediction output,Future target values indexed by forecast perio...,Forecast output; horizon remains a separate co...


,status
forecast_horizon,open
forecast_origin,open
allowed_history,open
backtesting_design,open
final_holdout_boundary,open
multi_step_strategy,open
baseline,open
model_family,open
target_transformation,open
evaluation_metrics,open


,temperature
period,
1920-01,40.6
1920-02,40.8
1920-03,44.4
1920-04,46.7
1920-05,54.1


## 6. Dataset Structure, Data Types, Units, and Domain Validity

The acquired source has two **transport columns** and 240 rows: `time` carries the original R time-series coordinate and `value` carries the observed temperature measurement. After the canonical reconstruction performed above, the analytical object for forecasting is not a two-feature table but a single numeric `pandas.Series` named `temperature`, indexed by monthly `PeriodIndex` values.

This distinction prevents the physical acquisition format from leaking into the modeling semantics:

- `time` is a numeric source coordinate used to authenticate and reconstruct temporal identity; it is **not** an ordinary predictor;
- `value` is numeric source data and becomes the canonical endogenous target `temperature` without changing its values;
- the analytical dataset contains one endogenous series and zero source exogenous predictors;
- the canonical target index is monthly (`M`) and contains 240 observation periods; and
- no categorical, text, identifier, or source-provided predictor fields belong to the forecasting problem.

The authenticated source unit is **degrees Fahrenheit**. This unit is part of the target contract and must remain attached to observed and forecast values on the public/original scale. No conversion to Celsius, normalization, standardization, or other unit/scale transformation is performed in this exploratory step.

Domain validation is intentionally conservative. For finite observed values, the only universal physical bound enforced here is that an absolute temperature expressed in degrees Fahrenheit cannot be below **absolute zero (`-459.67 °F`)**. No empirical Nottingham-specific lower or upper threshold is imposed, because deriving admissibility limits from the observed historical range would incorrectly turn sample behavior into a source-domain rule.

This section does not decide whether missing or non-finite observations exist, quantify the observed target range, classify extreme values as anomalies, or infer climate-specific outliers. Those questions belong to the dedicated missing/invalid-value, distribution, and anomaly sections that follow.

In [5]:
ABSOLUTE_ZERO_F = -459.67
EXPECTED_RAW_COLUMNS = ("time", "value")

if tuple(source_data.columns) != EXPECTED_RAW_COLUMNS:
    raise ValueError(
        "Unexpected raw transport structure: "
        f"expected {EXPECTED_RAW_COLUMNS}, "
        f"observed {tuple(source_data.columns)}."
    )

if source_data.shape != (EXPECTED_OBSERVATIONS, len(EXPECTED_RAW_COLUMNS)):
    raise ValueError(
        "Unexpected raw dataset shape: "
        f"expected {(EXPECTED_OBSERVATIONS, len(EXPECTED_RAW_COLUMNS))}, "
        f"observed {source_data.shape}."
    )

if not pd.api.types.is_numeric_dtype(source_data["time"]):
    raise TypeError("The raw source time coordinate must be numeric.")

if not pd.api.types.is_numeric_dtype(source_data["value"]):
    raise TypeError("The raw source value column must be numeric.")

if not isinstance(target_series, pd.Series):
    raise TypeError("The canonical forecasting target must be a pandas Series.")

if target_series.name != TARGET_NAME:
    raise ValueError(
        f"Expected canonical target name {TARGET_NAME!r}, "
        f"observed {target_series.name!r}."
    )

if len(target_series) != EXPECTED_OBSERVATIONS:
    raise ValueError(
        "Unexpected canonical target length: "
        f"expected {EXPECTED_OBSERVATIONS}, observed {len(target_series)}."
    )

if not pd.api.types.is_numeric_dtype(target_series.dtype):
    raise TypeError("The canonical forecasting target must remain numeric.")

if not isinstance(target_series.index, pd.PeriodIndex):
    raise TypeError("The canonical forecasting target must use a PeriodIndex.")

if target_series.index.freqstr != EXPECTED_FREQUENCY:
    raise ValueError(
        "Unexpected canonical target frequency: "
        f"expected {EXPECTED_FREQUENCY!r}, "
        f"observed {target_series.index.freqstr!r}."
    )

if forecasting_target_contract.target_unit != TARGET_UNIT:
    raise ValueError(
        "Target unit changed after contract definition: "
        f"expected {TARGET_UNIT!r}, "
        f"observed {forecasting_target_contract.target_unit!r}."
    )

target_numeric = target_series.to_numpy(dtype=float, copy=True)
finite_target_mask = np.isfinite(target_numeric)
below_absolute_zero_mask = (
    finite_target_mask & (target_numeric < ABSOLUTE_ZERO_F)
)
absolute_zero_violation_count = int(below_absolute_zero_mask.sum())

if absolute_zero_violation_count:
    raise ValueError(
        "Finite target values violate the universal Fahrenheit domain: "
        f"{absolute_zero_violation_count} observation(s) are below "
        f"absolute zero ({ABSOLUTE_ZERO_F} °F)."
    )

dataset_structure = pd.DataFrame(
    [
        ("Raw observations", len(source_data), "Source transport rows"),
        (
            "Raw transport columns",
            len(source_data.columns),
            "time coordinate + observed value",
        ),
        (
            "Raw time dtype",
            str(source_data["time"].dtype),
            "Numeric fractional-year source coordinate",
        ),
        (
            "Raw value dtype",
            str(source_data["value"].dtype),
            "Numeric source temperature value",
        ),
        (
            "Canonical analytical object",
            type(target_series).__name__,
            "One endogenous time series",
        ),
        (
            "Canonical target",
            target_series.name,
            TARGET_SEMANTICS,
        ),
        (
            "Canonical target dtype",
            str(target_series.dtype),
            "Numeric",
        ),
        (
            "Canonical index type",
            type(target_series.index).__name__,
            "Calendar-period identity",
        ),
        (
            "Canonical frequency",
            target_series.index.freqstr,
            "Monthly",
        ),
        (
            "Source exogenous predictors",
            forecasting_target_contract.source_exogenous_predictors,
            "None",
        ),
        (
            "Target unit",
            forecasting_target_contract.target_unit,
            "Original/public target scale",
        ),
        (
            "Finite-value physical domain",
            f">= {ABSOLUTE_ZERO_F} °F",
            "Universal absolute-temperature lower bound",
        ),
        (
            "Physical-domain violations",
            absolute_zero_violation_count,
            "Finite values below absolute zero",
        ),
    ],
    columns=["Property", "Observed", "Interpretation"],
)

display(dataset_structure)

,Property,Observed,Interpretation
0,Raw observations,240,Source transport rows
1,Raw transport columns,2,time coordinate + observed value
2,Raw time dtype,float64,Numeric fractional-year source coordinate
3,Raw value dtype,float64,Numeric source temperature value
4,Canonical analytical object,Series,One endogenous time series
5,Canonical target,temperature,Monthly average air temperature at Nottingham ...
6,Canonical target dtype,float64,Numeric
7,Canonical index type,PeriodIndex,Calendar-period identity
8,Canonical frequency,M,Monthly
9,Source exogenous predictors,0,None


## 7. Temporal Ordering, Missing Periods, and Timestamp Integrity

The canonical time index must represent one and only one observation period for every expected calendar month, in source order, with no temporal reordering or silent gap repair. The reconstruction validated in Section 4 already establishes the expected monthly grid from `1920-01` through `1939-12`; this section makes the corresponding integrity conditions explicit and fail-closed.

Temporal validity requires all of the following:

- the raw fractional-year source coordinate is strictly increasing in row order;
- the canonical `PeriodIndex` is strictly chronological and unique;
- the canonical target index remains exactly aligned with the reconstructed source index;
- consecutive canonical periods advance by exactly one month;
- no expected month is absent from the authenticated 1920–1939 coverage;
- no unexpected period exists outside that coverage; and
- no duplicate canonical period is present.

These checks are structural. They do **not** inspect whether target values are missing or non-finite; that is the responsibility of the next section. Likewise, the absence of duplicate canonical periods is established here as a timestamp-integrity invariant, while repeated temperature values and the scientific distinction between repeated measurements and source revisions remain for the dedicated duplicate/revision analysis.

Because the canonical index is a monthly `PeriodIndex`, timestamp integrity does not require a day, clock time, UTC offset, or timezone. Adding any of those would introduce temporal precision that is not present in the source semantics.

No missing period is imputed or synthesized by this check. If a future acquisition deviates from the authenticated monthly grid, the notebook must fail rather than silently repair the sequence.

In [6]:
raw_time_numeric = source_data["time"].to_numpy(dtype=float, copy=True)

raw_time_step = np.diff(raw_time_numeric)
raw_time_strictly_increasing = bool(np.all(raw_time_step > 0))

canonical_index = target_series.index

canonical_index_monotonic = bool(canonical_index.is_monotonic_increasing)
canonical_index_unique = bool(canonical_index.is_unique)

duplicate_period_mask = canonical_index.duplicated(keep=False)
duplicate_periods = canonical_index[duplicate_period_mask].unique()

full_expected_index = pd.period_range(
    start=EXPECTED_START_PERIOD,
    end=EXPECTED_END_PERIOD,
    freq=EXPECTED_FREQUENCY,
    name=canonical_index.name,
)

missing_periods = full_expected_index.difference(canonical_index)
unexpected_periods = canonical_index.difference(full_expected_index)

canonical_month_steps = np.diff(canonical_index.asi8)
monthly_step_integrity = bool(
    len(canonical_month_steps) == max(len(canonical_index) - 1, 0)
    and np.all(canonical_month_steps == 1)
)

target_index_aligned = bool(
    canonical_index.equals(canonical_source_series.index)
)

temporal_integrity_failures = []

if not raw_time_strictly_increasing:
    temporal_integrity_failures.append(
        "raw source time coordinate is not strictly increasing"
    )

if not canonical_index_monotonic:
    temporal_integrity_failures.append(
        "canonical PeriodIndex is not monotonically increasing"
    )

if not canonical_index_unique:
    temporal_integrity_failures.append(
        "canonical PeriodIndex contains duplicate periods"
    )

if not monthly_step_integrity:
    temporal_integrity_failures.append(
        "canonical PeriodIndex does not advance by exactly one month"
    )

if len(missing_periods):
    temporal_integrity_failures.append(
        f"{len(missing_periods)} expected monthly period(s) are missing"
    )

if len(unexpected_periods):
    temporal_integrity_failures.append(
        f"{len(unexpected_periods)} unexpected period(s) are present"
    )

if not target_index_aligned:
    temporal_integrity_failures.append(
        "canonical target index is not aligned with the reconstructed source index"
    )

if len(canonical_index) != EXPECTED_OBSERVATIONS:
    temporal_integrity_failures.append(
        "canonical observation count differs from the authenticated source count"
    )

if canonical_index[0] != EXPECTED_START_PERIOD:
    temporal_integrity_failures.append(
        "canonical start period differs from the authenticated source start"
    )

if canonical_index[-1] != EXPECTED_END_PERIOD:
    temporal_integrity_failures.append(
        "canonical end period differs from the authenticated source end"
    )

if temporal_integrity_failures:
    raise ValueError(
        "Temporal integrity validation failed: "
        + "; ".join(temporal_integrity_failures)
        + "."
    )

temporal_integrity_summary = pd.DataFrame(
    [
        (
            "Raw source order",
            raw_time_strictly_increasing,
            "Strictly increasing fractional-year coordinate",
        ),
        (
            "Canonical order",
            canonical_index_monotonic,
            "Monthly periods increase chronologically",
        ),
        (
            "Canonical uniqueness",
            canonical_index_unique,
            "Exactly one canonical observation per period",
        ),
        (
            "One-month step integrity",
            monthly_step_integrity,
            "Every adjacent canonical period advances by one month",
        ),
        (
            "Missing expected periods",
            len(missing_periods),
            "Expected monthly periods absent from canonical coverage",
        ),
        (
            "Unexpected periods",
            len(unexpected_periods),
            "Canonical periods outside authenticated coverage",
        ),
        (
            "Duplicate canonical periods",
            len(duplicate_periods),
            "Repeated monthly identities",
        ),
        (
            "Target/index alignment",
            target_index_aligned,
            "Target preserves reconstructed temporal identity",
        ),
        (
            "Canonical start",
            str(canonical_index[0]),
            f"Expected {EXPECTED_START_PERIOD}",
        ),
        (
            "Canonical end",
            str(canonical_index[-1]),
            f"Expected {EXPECTED_END_PERIOD}",
        ),
        (
            "Canonical observations",
            len(canonical_index),
            f"Expected {EXPECTED_OBSERVATIONS}",
        ),
        (
            "Timezone applicability",
            "not applicable",
            "Monthly PeriodIndex represents calendar periods, not instants",
        ),
    ],
    columns=["Check", "Observed", "Interpretation"],
)

display(temporal_integrity_summary)

,Check,Observed,Interpretation
0,Raw source order,True,Strictly increasing fractional-year coordinate
1,Canonical order,True,Monthly periods increase chronologically
2,Canonical uniqueness,True,Exactly one canonical observation per period
3,One-month step integrity,True,Every adjacent canonical period advances by on...
4,Missing expected periods,0,Expected monthly periods absent from canonical...
5,Unexpected periods,0,Canonical periods outside authenticated coverage
6,Duplicate canonical periods,0,Repeated monthly identities
7,Target/index alignment,True,Target preserves reconstructed temporal identity
8,Canonical start,1920-01,Expected 1920-01
9,Canonical end,1939-12,Expected 1939-12


## 8. Missing, Invalid, and Non-Finite Values

## 9. Duplicate Timestamps, Repeated Values, and Source Revision Semantics

## 10. Target Distribution, Range, and Level Summary

## 11. Time-Series Evolution and Long-Term Trend Signals

## 12. Seasonal Structure and Calendar-Month Profiles

## 13. Decomposition and Trend/Seasonal Strength

## 14. Autocorrelation and Lag Dependence

## 15. Stationarity and Transformation/Differencing Signals

## 16. Outliers, Anomalies, and Structural-Break Signals

## 17. Forecast Horizon and Final-Holdout Feasibility

## 18. Forecasting Baselines and Forecastability Considerations

## 19. Temporal Leakage and Evaluation-Boundary Risks

## 20. Key Exploratory Insights

## 21. Preparation and Backtesting Decisions

## 22. Exploration Handoff and Next Steps